In [1]:
import torch
import torch.nn as nn
from PIL import Image
from typing import Tuple

In [2]:
from utils.config import (
    MODEL_NAME, NUM_CLASSES, DEVICE, CHECKPOINT_PATH, CHECKPOINT_PATH_1
)

In [3]:
import timm 

In [4]:
def get_model(model_name: str, num_classes: int, pretrained: bool = False) -> nn.Module:
    """Instantiates the Vision Transformer (ViT) model structure from timm."""
    # Note: We set pretrained=False here because we are loading weights from a file
    model = timm.create_model(
        model_name, 
        pretrained=pretrained, 
        num_classes=num_classes
    )
    return model.to(DEVICE)


In [5]:
def load_best_model(checkpoint_path: str) -> nn.Module | None:
    """Loads the model architecture and applies the saved weights."""
    
    # 1. Instantiate the model structure
    model = get_model(
        MODEL_NAME, 
        NUM_CLASSES, 
        pretrained=False
    )
    
    # 2. Load the checkpoint file (map_location handles loading GPU model on CPU if necessary)
    try:
        checkpoint = torch.load(
            checkpoint_path, 
            map_location=DEVICE,
            weights_only=True)
    except FileNotFoundError:
        print(f"ERROR: Checkpoint file not found at {checkpoint_path}")
        return None
    
    # 3. Apply the saved weights to the model structure
    model.load_state_dict(checkpoint['model_state'])
    
    # 4. Set model to evaluation mode (crucial for disabling dropout/batch norm tracking)
    model.eval()
    
    print(f"Model ({MODEL_NAME}) successfully loaded.")
    print(f"Best Val AUC achieved during training: {checkpoint.get('best_val_auc', 'N/A')}")
    return model

In [6]:
loaded_model   = load_best_model(CHECKPOINT_PATH)
loaded_model_1 = load_best_model(CHECKPOINT_PATH_1)

Model (vit_base_patch16_224) successfully loaded.
Best Val AUC achieved during training: 1.0
Model (vit_base_patch16_224) successfully loaded.
Best Val AUC achieved during training: 0.9861549419241726


In [7]:
from utils.dataset import val_tf
import numpy as np

def check_pneumonia(model: nn.Module, image_path: str) -> Tuple[str, float]:
    """
    Predicts whether a chest X-ray contains signs of pneumonia.
    
    Returns: A tuple (Prediction Label, Confidence Score)
    """
    
    if model is None:
        return "Model Not Loaded", 0.0

    # 1. Load and Preprocess Image
    try:
        # Load and convert to RGB (as required by ViT)
        img = Image.open(image_path).convert('RGB')
        
        # Apply inference transforms
        tensor_img = val_tf(img)
        
        # Add batch dimension: [C, H, W] -> [1, C, H, W]
        input_tensor = tensor_img.unsqueeze(0).to(DEVICE)
        
    except FileNotFoundError:
        return f"Error: Image not found at {image_path}", 0.0
    except Exception as e:
        return f"Error during processing: {e}", 0.0

    # 2. Run Inference
    with torch.no_grad():
        # Set the model to evaluation mode again (safety)
        model.eval()
        
        # Use autocast for consistency, though not strictly needed for inference
        with torch.autocast(device_type=DEVICE, dtype=torch.float16): 
            logits = model(input_tensor)
        
        # Get probabilities
        probabilities = torch.softmax(logits, dim=1).cpu().numpy()[0]
        
    # 3. Determine Prediction (THRESHOLD LOGIC)
    probabilities = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pneumonia_prob = probabilities[1]
    normal_prob = probabilities[0]
    
    # Standard method: The class with the highest probability is the prediction
    predicted_class_index = np.argmax(probabilities)
    prediction_label = "PNEUMONIA" if predicted_class_index == 1 else "NORMAL"
    
    # The confidence is usually reported for the positive class (Pneumonia)
    # If the prediction is NORMAL, confidence is (1 - Pneumonia_Prob)
    confidence_score = float(pneumonia_prob)

    return prediction_label, confidence_score

In [8]:
base_path = "C:\\Users\\HP\\Desktop\\SLIIT\\Y4 SEM 1\\DL\\Ass\\Assignment\\DL-project\\ViT\\dataset\\chest_xray"

check_image_path   = base_path + "\\val\\NORMAL\\NORMAL2-IM-1436-0001.jpeg"
check_image_path_1 = base_path + "\\val\\PNEUMONIA\\person1950_bacteria_4881.jpeg" 
check_image_path_2 = base_path + "\\test\\NORMAL\\IM-0001-0001.jpeg"
check_image_path_3 = base_path + "\\test\\PNEUMONIA\\person1_virus_6.jpeg"

# pred, conf = check_pneumonia(loaded_model, check_image_path_2)

pred, conf = check_pneumonia(loaded_model_1, check_image_path_2)

# print(f"Prediction: {pred}, Confidence (Pneumonia): {conf:.4f}") 

print(f"Prediction: {pred}")
if pred == "PNEUMONIA":
    print(f"Confidence score: {conf}")

Prediction: NORMAL


In [9]:
import pandas as pd
from pathlib import Path

In [12]:
from utils.config import CLASS_NAMES
from typing import List

def evaluate_folder(model: nn.Module, folder_path: str) -> Tuple[pd.DataFrame, float,int,int]:
    """
    Loops through all images in a given folder (and its subdirectories), 
    evaluates them, and calculates overall accuracy.
    
    Args:
        model: The loaded PyTorch model in model.eval() mode.
        folder_path: The root directory to search for images (e.g., 'path/to/test').
        inference_tf: The image preprocessing pipeline.
        
    Returns:
        A tuple containing: (DataFrame of all results, Overall Accuracy)
    """
    if model is None:
        print("Error: Model not loaded. Cannot proceed with evaluation.")
        return pd.DataFrame(), 0.0
    
    root_path = Path(folder_path)
    all_results = []
    
    # Define the reverse class map for printing
    label_to_class = {v: k for k, v in CLASS_NAMES.items()}
    
    # Search for common image extensions recursively
    image_files: List[Path] = list(root_path.rglob("*.jpeg")) + \
                              list(root_path.rglob("*.png")) + \
                              list(root_path.rglob("*.jpg"))
    
    if not image_files:
        print(f"No image files found in {root_path}")
        return pd.DataFrame(), 0.0

    print(f"Found {len(image_files)} images for evaluation. This may take a moment...")
    
    # Counters for accuracy calculation
    correct_predictions = 0
    total_samples = 0
    
    for image_file in image_files:
        # Determine the True Label from the parent directory name (e.g., 'NORMAL' or 'PNEUMONIA')
        true_label_str = image_file.parent.name.upper()
        true_label_int = CLASS_NAMES.get(true_label_str, -1) # -1 if not found

        if true_label_int == -1:
            print(f"Warning: Skipping file {image_file.name} with unknown parent class.")
            continue
            
        # --- PREDICTION ---
        try:
            # Load and preprocess image (same logic as check_pneumonia, but streamlined)
            img = Image.open(image_file).convert('RGB')
            tensor_img = val_tf(img)
            input_tensor = tensor_img.unsqueeze(0).to(DEVICE)

            with torch.no_grad():
                with torch.autocast(device_type=DEVICE, dtype=torch.float16):
                    logits = model(input_tensor)
                
                probabilities = torch.softmax(logits, dim=1).cpu().numpy()[0]
                pred_label_int = torch.argmax(logits, dim=1).item()
                
        except Exception as e:
            print(f"Error processing {image_file.name}: {e}. Skipping.")
            continue
        
        # --- RESULTS ---
        pred_label_str = label_to_class[pred_label_int]
        pneumonia_conf = probabilities[CLASS_NAMES['PNEUMONIA']]
        normal_conf = probabilities[CLASS_NAMES['NORMAL']]

        is_correct = (pred_label_int == true_label_int)
        
        if is_correct:
            correct_predictions += 1
            
        total_samples += 1
        
        all_results.append({
            'Filename': image_file.name,
            'True Label': true_label_str,
            'Predicted Label': pred_label_str,
            'Normal Confidence': f"{normal_conf:.4f}",
            'Pneumonia Confidence': f"{pneumonia_conf:.4f}",
            'Correct': 'Yes' if is_correct else 'No'
        })

    # --- FINAL SUMMARY ---
    df_results = pd.DataFrame(all_results)
    overall_accuracy = correct_predictions / total_samples if total_samples > 0 else 0.0
    
    return df_results, overall_accuracy, correct_predictions, total_samples

In [13]:
test_folder_path = base_path + "\\test"
val_folder_path  = base_path + "\\val"

# df_final, final_acc, correct_predictions, total_samples = evaluate_folder(loaded_model, val_folder_path)

df_final_1, final_acc_1, correct_predictions_1, total_samples_1 = evaluate_folder(loaded_model_1, test_folder_path)

# print("\n--- DETAILED PREDICTIONS MODEL 1 ---")
# display(df_final)
# print(f"\n--- OVERALL ACCURACY on Folder: {final_acc:.4f} ({correct_predictions}/{total_samples}) ---")

print("\n--- DETAILED PREDICTIONS MODEL 2 ---")
display(df_final_1)
print(f"\n--- OVERALL ACCURACY on Folder: {final_acc_1:.4f} ({correct_predictions_1}/{total_samples_1}) ---")

Found 624 images for evaluation. This may take a moment...

--- DETAILED PREDICTIONS MODEL 2 ---


,Filename,True Label,Predicted Label,Normal Confidence,Pneumonia Confidence,Correct
0,IM-0001-0001.jpeg,NORMAL,NORMAL,0.9873,0.0129,Yes
1,IM-0003-0001.jpeg,NORMAL,NORMAL,0.9937,0.0066,Yes
2,IM-0005-0001.jpeg,NORMAL,NORMAL,0.8940,0.1060,Yes
3,IM-0006-0001.jpeg,NORMAL,NORMAL,0.8750,0.1251,Yes
4,IM-0007-0001.jpeg,NORMAL,NORMAL,0.6729,0.3274,Yes
...,...,...,...,...,...,...
619,person96_bacteria_465.jpeg,PNEUMONIA,PNEUMONIA,0.0000,1.0000,Yes
620,person96_bacteria_466.jpeg,PNEUMONIA,PNEUMONIA,0.0000,1.0000,Yes
621,person97_bacteria_468.jpeg,PNEUMONIA,PNEUMONIA,0.0000,1.0000,Yes
622,person99_bacteria_473.jpeg,PNEUMONIA,PNEUMONIA,0.0000,1.0000,Yes



--- OVERALL ACCURACY on Folder: 0.8045 (502/624) ---


In [14]:
# --- Save as a CSV ---
# output_filename = 'ViT_Evaluation_Results.csv'
output_filename_1 = 'ViT_Evaluation_Results_1.csv'

# df_final.to_csv(output_filename, index=False) 
df_final_1.to_csv(output_filename_1, index=False)

print(f"\n✅ Results successfully saved to {output_filename_1}")


✅ Results successfully saved to ViT_Evaluation_Results_1.csv
